# Phase 4 — Optionality Experiment

Per the aims doc, this is the project's most interview-worthy result: it directly ties Phase 1's forecasting work to a trading decision. For a small set of illustrative cargoes, compare two strategies using `src/optionality.py`:

- **Commit-now**: pick the destination today (best netback on currently-known prices), lock it in, realize whatever netback results at delivery (1 month out).
- **Wait-and-redirect**: hold the decision open for 1 week, then choose based on that week's realized prices, still deliver at the same final date.

$$V_{option} = \mathbb{E}[\text{Payoff}_{flexible}] - \mathbb{E}[\text{Payoff}_{committed}]$$

Both strategies decide using **netback** (not headline price) -- unlike Phase 3's naive baseline, which was deliberately naive on purpose. Here we're comparing two *rational* strategies against each other, not rational vs. naive. Capacity constraints are intentionally not enforced -- this is a single-cargo marginal analysis ("what is flexibility worth on one cargo"), not a portfolio allocation problem; that's Phase 3's job.

In [ ]:
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from netback import compute_netback_all_destinations
from network_config import ORIGINS, VESSEL
from optionality import evaluate_optionality, simulate_decision_scenarios
from simulator import HubParams

## Simulate decision scenarios

Weekly-resolution paths out to 4 weeks (1 month), re-expressed from the monthly calibration via `rescale_dt()` -- theta/sigma carry over as continuous-time rates, jump_prob is rescaled to preserve the implied annual jump frequency. Correlation structure is assumed unchanged across frequencies (no weekly data exists to estimate it separately).

In [ ]:
with open("../data/processed/calibration/hub_params.json") as f:
    calibration = json.load(f)
hub_order = calibration["correlation"]["hub_order"]
corr_matrix = calibration["correlation"]["matrix"]
all_params = [HubParams(**calibration["primary"][name]) for name in hub_order]

with open("../data/processed/combined/monthly_aligned.csv") as f:
    rows = list(csv.DictReader(f))
latest = rows[-1]
start_prices = {"HENRY_HUB": float(latest["henry_hub"]), "TTF": float(latest["ttf"]), "JKM": float(latest["jkm"])}
start_freight = {"FREIGHT_ATLANTIC": float(latest["freight_atlantic"]), "FREIGHT_PACIFIC": float(latest["freight_pacific"])}
start_all = {**start_prices, **start_freight}

N_SCENARIOS = 3000
scenarios = simulate_decision_scenarios(all_params, corr_matrix, start_all, n_scenarios=N_SCENARIOS, seed=99)
print(f"{N_SCENARIOS} weekly-resolution scenarios simulated, decision at week 1, delivery at week 4")

## Cargo set: 2 illustrative cargoes per origin

Optionality value is driven entirely by which destinations an origin can reach and their route economics, not by anything cargo-specific -- so cargoes from the same origin have (up to Monte Carlo noise) the same option value. The "6 cargoes" here represent 2 loadings per origin during the month; in practice this scales linearly to all of that origin's cargoes (e.g. US_GC's full ~8/month).

In [ ]:
CARGOES_PER_ORIGIN = 2
cargo_size = VESSEL["cargo_size_mmbtu"]

results = {}
for origin in ORIGINS:
    results[origin] = evaluate_optionality(origin, scenarios, start_prices, start_freight)

rows_out = []
for origin, r in results.items():
    rows_out.append({
        "origin": origin,
        "committed_destination": r["committed_destination"],
        "redirect_rate": f"{r['redirect_rate']:.1%}",
        "V_option ($/MMBtu)": round(r["v_option_usd_per_mmbtu"], 3),
        "V_option ($/cargo)": f"${r['v_option_usd_per_mmbtu'] * cargo_size:,.0f}",
        f"V_option, {CARGOES_PER_ORIGIN} cargoes ($)": f"${r['v_option_usd_per_mmbtu'] * cargo_size * CARGOES_PER_ORIGIN:,.0f}",
    })
pd.DataFrame(rows_out)

## Why the option is worth more for some origins than others

In [ ]:
for origin, r in results.items():
    n_dest = len({dest.destination for dest in compute_netback_all_destinations(origin, start_prices, start_freight)})
    print(f"{origin:<10} {n_dest} reachable destinations, committed={r['committed_destination']:<12} "
          f"redirects in {r['redirect_rate']:.1%} of scenarios, V_option=${r['v_option_usd_per_mmbtu']:.3f}/MMBtu")

print("\nUS_GC has the most reachable destinations (3, incl. domestic) and the highest redirect rate/option value.")
print("Australia has the lowest -- its Europe route is so much more expensive (long Cape voyage) that")
print("price moves rarely flip the decision away from Asia, so there's less genuine ambiguity to exploit.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (origin, r) in zip(axes, results.items()):
    ax.hist(r["payoff_committed"], bins=40, alpha=0.5, label="committed", density=True)
    ax.hist(r["payoff_flexible"], bins=40, alpha=0.5, label="flexible", density=True)
    ax.axvline(r["payoff_committed"].mean(), color="C0", linestyle="--")
    ax.axvline(r["payoff_flexible"].mean(), color="C1", linestyle="--")
    ax.set_title(f"{origin}\nV_option=${r['v_option_usd_per_mmbtu']:.3f}/MMBtu")
    ax.set_xlabel("delivery-time netback ($/MMBtu)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")
fig.tight_layout()
plt.show()

## Relating V_option back to calibration quality

The aims doc specifically asks to connect this back to how tight/well-calibrated the price intervals are (Phase 1's coverage checks). This experiment's option value comes almost entirely from the **TTF-JKM spread's** volatility over one week, since that's what determines whether the Europe/Asia choice flips (US_GC's domestic option adds a third leg, but Henry Hub barely moves relative to the export legs).

Phase 1's coverage checks found TTF and JKM both landing in a reasonable 79-89% range against a 90% target -- calibration good enough to trust this option value as directionally right. That's *not* true for a freight-driven optionality question (e.g. "wait to see which basin's freight rate is cheaper"): freight's coverage check came in at only ~65% against the same target, a direct sign the model under-states freight's real volatility. If this experiment's flip were being driven by freight variation instead of the TTF-JKM spread, the option value here would likely be **understated**, not overstated -- worth flagging explicitly rather than presenting `V_option` as more precise than the underlying calibration supports.

## Save

In [ ]:
import os

os.makedirs("../data/processed/optionality", exist_ok=True)
summary = {
    origin: {
        "committed_destination": r["committed_destination"],
        "redirect_rate": r["redirect_rate"],
        "v_option_usd_per_mmbtu": r["v_option_usd_per_mmbtu"],
        "v_option_usd_per_cargo": r["v_option_usd_per_mmbtu"] * cargo_size,
    }
    for origin, r in results.items()
}
with open("../data/processed/optionality/v_option.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved ../data/processed/optionality/v_option.json")